# WLA / DAP Historical Curve Reconstruction

This notebook reconstructs the historical Bloomberg `WLA Index` futures chain used as the DAP / ID x IPCA real-rate curve input for the thesis.

**Thesis sample:** 2010-01-29 to 2025-06-30, month-end observations.

The notebook requires Bloomberg BQuant/BQL access. A reviewer without Bloomberg can independently validate the underlying data using B3's historical **BVBG.086.01 PriceReport** files:

https://b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/boletins-diarios/pesquisa-por-pregao/pesquisa-por-pregao/

See `README_WLA_DAP_REPLICATION.md` for the public replication route and methodology notes.

## Bloomberg Excel equivalent

For a single historical date, Bloomberg Excel can return the live futures chain with:

```excel
=BDS("WLA Index","FUT_CHAIN","CHAIN_DATE=20240131","INCLUDE_EXPIRED_CONTRACTS=N")
```

Change the date as required.

Do **not** assume that the historical chain always contains the same number of contracts.

In [ ]:
import bql
import pandas as pd

bq = bql.Service()

## Fetch the historical WLA futures chain for one date

The procedure:

1. retrieves the historical live WLA futures universe;
2. retrieves specific contract ticker, `PX_LAST`, and `FUTURES_VALUATION_DATE`;
3. sorts contracts by the actual valuation date;
4. assigns a chain position only after the true ordering is established.

Missing `PX_LAST` values are retained. Zero and negative rates are also retained because they can be valid real-rate observations.

In [ ]:
def fetch_wla_chain(d):
    # Convert date to BQL format
    d = pd.Timestamp(d).strftime("%Y-%m-%d")

    # Historical WLA futures universe as of date d
    universe = bq.univ.futures(
        "WLA Index",
        dates=d
    )

    # Data required for the thesis curve
    items = {
        "Ticker": bq.data.id(),
        "PX_LAST": bq.data.px_last(dates=d),
        "FUTURES_VALUATION_DATE": bq.data.futures_valuation_date(),
    }

    # Execute BQL request
    resp = bq.execute(
        bql.Request(universe, items)
    )

    # Combine BQL responses
    out = pd.concat(
        [x.df() for x in resp],
        axis=1
    )

    # Keep only relevant columns
    out = out[
        [
            "Ticker",
            "PX_LAST",
            "FUTURES_VALUATION_DATE",
        ]
    ].copy()

    # Convert valuation date to datetime
    out["FUTURES_VALUATION_DATE"] = pd.to_datetime(
        out["FUTURES_VALUATION_DATE"]
    )

    # Order contracts from shortest to longest actual maturity
    out = (
        out
        .sort_values("FUTURES_VALUATION_DATE")
        .reset_index(drop=True)
    )

    # Assign position BEFORE doing anything with missing PX_LAST values
    out["CHAIN_POSITION"] = range(
        1,
        len(out) + 1
    )

    # Historical as-of date
    out["CHAIN_DATE"] = pd.to_datetime(d)

    return out[
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "Ticker",
            "FUTURES_VALUATION_DATE",
            "PX_LAST",
        ]
    ]

## Thesis month-end dates

This reproduces the 186 business month-end observations used for the historical reconstruction.

In [ ]:
curve_dates = pd.date_range(
    start="2010-01-01",
    end="2025-06-30",
    freq="BM"
)

print("Number of curve dates:", len(curve_dates))
print("First date:", curve_dates[0])
print("Last date:", curve_dates[-1])

## Retrieve all historical chains

Each date is requested separately because the live contract universe changes over time.

In [ ]:
all_results = []
errors = []

for i, d in enumerate(curve_dates, start=1):
    try:
        chain = fetch_wla_chain(d)
        all_results.append(chain)

        print(
            f"[{i}/{len(curve_dates)}] "
            f"{d.strftime('%Y-%m-%d')} "
            f"-> {len(chain)} contracts"
        )

    except Exception as e:
        errors.append(
            {
                "CHAIN_DATE": d,
                "ERROR": str(e),
            }
        )

        print(
            f"[{i}/{len(curve_dates)}] "
            f"{d.strftime('%Y-%m-%d')} "
            f"-> ERROR: {e}"
        )

full_chain = pd.concat(
    all_results,
    ignore_index=True
)

errors_df = pd.DataFrame(errors)

print("\nFinished.")
print("Curve dates requested:", len(curve_dates))
print(
    "Curve dates retrieved:",
    full_chain["CHAIN_DATE"].nunique()
)
print(
    "Total contract rows:",
    len(full_chain)
)
print(
    "Errors:",
    len(errors_df)
)

## Audit the raw quotes

The raw extraction should not impose a positivity filter.

Positive, zero and negative DAP rates can all be legitimate observations. Only `NaN` represents an unavailable quote at this stage.

In [ ]:
print("Total rows:", len(full_chain))
print("PX_LAST missing:", full_chain["PX_LAST"].isna().sum())
print("PX_LAST equal to zero:", (full_chain["PX_LAST"] == 0).sum())
print("PX_LAST negative:", (full_chain["PX_LAST"] < 0).sum())
print("PX_LAST positive:", (full_chain["PX_LAST"] > 0).sum())

print("\nDates with negative WLA rates:")
print(
    full_chain.loc[
        full_chain["PX_LAST"] < 0,
        [
            "CHAIN_DATE",
            "Ticker",
            "PX_LAST",
            "FUTURES_VALUATION_DATE",
        ]
    ].to_string(index=False)
)

print("\nZero WLA rates:")
print(
    full_chain.loc[
        full_chain["PX_LAST"] == 0,
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "Ticker",
            "PX_LAST",
            "FUTURES_VALUATION_DATE",
        ]
    ].to_string(index=False)
)

## Export the authoritative raw historical dataset

No tenor calculation, interpolation, unit conversion or missing-value imputation is performed here. Those transformations belong in the thesis Python pipeline.

Expected thesis-window extraction:

- 186 dates
- 3,143 rows
- 966 missing `PX_LAST`
- 2 zero rates
- 86 negative rates
- 2,089 positive rates

In [ ]:
export_df = (
    full_chain
    .sort_values(
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "FUTURES_VALUATION_DATE",
        ]
    )
    .reset_index(drop=True)
)

excel_file = "wla_historical_chain_2010_2025_raw.xlsx"
csv_file = "wla_historical_chain_2010_2025_raw.csv"

export_df.to_excel(
    excel_file,
    sheet_name="wla_raw",
    index=False
)

export_df.to_csv(
    csv_file,
    index=False
)

print("Export completed.")
print("Excel:", excel_file)
print("CSV:", csv_file)
print("Rows:", len(export_df))
print("Dates:", export_df["CHAIN_DATE"].nunique())
print("Missing PX_LAST:", export_df["PX_LAST"].isna().sum())
print("Zero PX_LAST:", (export_df["PX_LAST"] == 0).sum())
print("Negative PX_LAST:", (export_df["PX_LAST"] < 0).sum())

## B3 public validation note

A reviewer without Bloomberg can use B3's historical **Boletim de Negociação — BVBG.086.01 PriceReport** for the corresponding date and identify the DAP contract.

Example already cross-checked:

- Date: 2018-09-28
- Bloomberg contract: `WLV18 Index`
- Bloomberg `PX_LAST`: `0.000`
- B3 DAP V18 official adjustment rate: `0.000`

Therefore, zero observations must not automatically be classified as missing.